# 회귀 스위트

Python 3.12 새 커널에서 Before: 평균만 보는 기준의 누락부터 확인합니다.

## 개념을 코드에 연결하기

**호출 표기와 실행 시점.** 회귀 스위트는 고정 평가 사례와 통과 조건을 새 버전에 반복 적용합니다. 이 노트북은 사례별 품질 조건을 확인하고, 지연 p95와 비용 p99는 전체 스위트의 집계값을 한 번씩 확인합니다. 하나라도 실패하면 전체 통과 조건이 거짓이 됩니다.

**다른 상황에 적용.** 프롬프트를 바꾼 뒤 전체 평균 점수는 올랐지만 개인정보 질문 한 건의 안전 조건이 실패할 수 있습니다. 사례 태그별 필수 조건을 두면 전체 개선과 중요한 하위 집단의 퇴보를 함께 확인할 수 있습니다.

**비교와 경계.** 회귀 검사는 등록한 사례만 보호합니다. 통과했다고 새 유형의 오류가 없다는 뜻은 아닙니다. 운영에서 발견한 실패를 사례로 추가하고, 평가 데이터 자체의 변경과 시스템 변경을 별도 버전으로 관리해야 합니다.


## Python 3.12 패키지 버전 고정

**이 셀의 코드 읽기.** `%pip install`은 이 노트북 커널이 사용할 패키지를 설치합니다. 앞 셀의 값을 입력으로 사용하지 않으며, 설치가 끝난 뒤의 코드 셀에서 패키지를 가져와 사용합니다.


In [1]:
# 실습 환경에서 사용하는 보조 패키지 버전을 고정합니다.
%pip install -q python-dotenv==1.2.3

Note: you may need to restart the kernel to use updated packages.


## Before: 평균만 보는 기준의 누락

**이 셀의 코드 읽기.** `for` 블록은 입력 항목을 하나씩 처리하며, 반복이 끝난 뒤 모은 값이 다음 셀의 입력이 됩니다. `print()`는 확인용 문자열을 화면에 표시할 뿐 함수의 반환값을 바꾸지 않습니다.


In [2]:
# 평균 기준만 사용했을 때 Q090의 핵심 실패를 놓치는지 확인합니다.
baseline = {'Q002': 1, 'Q006': 1, 'Q090': 1}
current = {'Q002': 1, 'Q006': 1, 'Q090': 0}
average_pass = sum(current.values()) / len(current) >= 0.66
print('Before 평균 기준 통과:', average_pass)
print('놓친 핵심 실패:', [qid for qid in baseline if current[qid] < baseline[qid]])

Before 평균 기준 통과: True
놓친 핵심 실패: ['Q090']


## After: 사례별 품질·지연·비용 통과 조건

**이 셀의 코드 읽기.** `gate(scores: dict[str, int])`는 함수를 정의하는 표기입니다. 이 셀에서는 함수 객체만 준비되고 들여쓴 본문은 뒤에서 함수를 호출할 때 실행됩니다. `return` 뒤의 값은 화면 출력이 아니라 호출한 코드로 전달되는 반환값입니다. `print()`는 확인용 문자열을 화면에 표시할 뿐 함수의 반환값을 바꾸지 않습니다.


In [3]:
# 사례별 품질과 지연·비용 상한을 모두 검사하는 gate 함수를 정의합니다.
limits = {'p95_total_ms': 1500, 'p99_cost_usd': 0.08}
observed = {'p95_total_ms': 980, 'p99_cost_usd': 0.06}
def gate(scores: dict[str, int]) -> bool:
    quality = all(scores[qid] >= baseline[qid] for qid in baseline)
    latency = observed['p95_total_ms'] <= limits['p95_total_ms']
    cost = observed['p99_cost_usd'] <= limits['p99_cost_usd']
    return quality and latency and cost
print('After 사례별 기준 통과:', gate(current))

After 사례별 기준 통과: False


## 수정 결과를 같은 기준으로 재검사

**이 셀의 코드 읽기.** `print()`는 확인용 문자열을 화면에 표시할 뿐 함수의 반환값을 바꾸지 않습니다. `assert`는 조건이 참이면 출력 없이 통과하고 거짓이면 실행을 중단하므로, 바로 앞 결과를 검증하는 역할을 합니다.


In [4]:
# Q090을 수정한 결과가 같은 고정 기준을 통과하는지 검사합니다.
fixed = {'Q002': 1, 'Q006': 1, 'Q090': 1}
print('수정 결과 통과:', gate(fixed))
assert not gate(current)
assert gate(fixed)
print('회귀 데이터셋 버전: rag-regression-v1.0.0')

수정 결과 통과: True
회귀 데이터셋 버전: rag-regression-v1.0.0
